# 01 — Common Data Quality Audit

**ผู้รับผิดชอบ:** ทอฝัน  
**วันที่ตรวจ:** 18 สิงหาคม 2026  
**ขอบเขต:** ตรวจข้อมูลที่ผ่านการทำความสะอาดแล้วทั้ง 5 ตารางใน `data/processed/` โดยไม่แก้ไขไฟล์ต้นฉบับใน `data/raw/`

## เป้าหมายของ Notebook

1. ตรวจโครงสร้าง ค่าว่าง และคีย์ของแต่ละตาราง
2. ตรวจช่วงค่าและกฎทางธุรกิจที่ข้อมูลควรเป็นไปตาม
3. ตรวจความสัมพันธ์ระหว่างตารางและความถูกต้องของวันที่
4. เปรียบเทียบข้อมูลก่อนและหลังทำความสะอาด

ในตารางผลตรวจ `failures` คือจำนวนแถวที่ไม่ผ่านเงื่อนไข โดย `PASS` หมายถึงไม่พบแถวที่ผิดเงื่อนไข


## 0. เตรียม Notebook

Cell แรก import library และค้นหา project root เพื่อให้รันได้ทั้งจากโฟลเดอร์หลักและโฟลเดอร์ `notebooks/`


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    '''Return the nearest parent directory containing both data/ and src/.'''
    for candidate in (start, *start.parents):
        if (candidate / 'data').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('ไม่พบ project root ที่มีโฟลเดอร์ data/ และ src/')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from kaverentai.data.load_data import TABLES, load_all_data
from kaverentai.data.validate_data import validate_tables

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)


### ค่าคงที่ที่ใช้ในกฎตรวจสอบ

การตั้งชื่อค่าคงที่ช่วยหลีกเลี่ยงตัวเลขหรือหมวดหมู่ที่ไม่ทราบที่มา (magic values) ภายในโค้ดตรวจสอบ


In [ ]:
DATA_START_DATE = pd.Timestamp('2023-01-03')
DAYS_PER_WEEK = 7

VALID_ROOM_TYPES = {'studio', '1BR', '1BR Plus'}
VALID_VIEWS = {'standard', 'city', 'pool'}
VALID_SEASONS = {'normal', 'peak', 'shoulder', 'off'}

VALID_FLOOR_RANGE = (1, 8)
VALID_SIZE_RANGE_SQM = (20, 50)
RENT_REVIEW_RANGE = (5_000, 30_000)

# ข้อมูลจำลองนี้กำหนดให้สัญญา 6 และ 12 เดือนเท่ากับ 24 และ 48 สัปดาห์
LEASE_WEEKS_BY_MONTHS = {6: 24, 12: 48}


In [ ]:
tables = load_all_data(cleaned=True, data_dir=PROCESSED_DIR)
raw_tables = load_all_data(cleaned=False, data_dir=RAW_DIR)

# ตั้งชื่อตารางครั้งเดียว เพื่อให้ทุกหัวข้อด้านล่างอ้างอิงได้โดยตรง
projects = tables['projects']
units = tables['units']
listings = tables['listings']
leases = tables['leases']
weekly_market = tables['weekly_market']

print(f'Project root: {PROJECT_ROOT}')
print(f'Loaded tables: {list(tables)}')


### รูปแบบการบันทึกผลตรวจ

แต่ละกฎจะสร้าง Boolean mask ซึ่งมีค่า `True` ตรงแถวที่ไม่ผ่าน จากนั้น `record_check()` จะนับจำนวน `True` และกำหนดสถานะให้เป็น `PASS`, `WARN` หรือ `FAIL`


In [ ]:
quality_checks = []


def record_check(
    area: str,
    check: str,
    failure_mask,
    note: str = '',
    failure_status: str = 'FAIL',
) -> None:
    '''Add one data-quality result; True values in failure_mask mean failures.'''
    failure_count = int(np.sum(failure_mask))
    status = 'PASS' if failure_count == 0 else failure_status

    quality_checks.append({
        'area': area,
        'check': check,
        'failures': failure_count,
        'status': status,
        'note': note,
    })


## 1. ภาพรวมและ schema ขั้นพื้นฐาน

เริ่มจากตรวจว่าทุกตารางโหลดได้ มีข้อมูล และมีคอลัมน์บังคับตาม `validate_data.py`


In [ ]:
inventory_rows = []

for table_name, frame in tables.items():
    inventory_rows.append({
        'table': table_name,
        'rows': len(frame),
        'columns': len(frame.columns),
        'duplicate_rows': int(frame.duplicated().sum()),
        'memory_mb': frame.memory_usage(deep=True).sum() / 1024**2,
    })

inventory = pd.DataFrame(inventory_rows).set_index('table')
display(inventory.style.format({'rows': '{:,}', 'memory_mb': '{:.2f}'}))


In [ ]:
schema_errors = validate_tables(tables)
schema_failed = len(schema_errors) > 0

record_check(
    area='Schema',
    check='ทุกตารางมีข้อมูลและมีคอลัมน์บังคับ',
    failure_mask=schema_failed,
    note='; '.join(schema_errors),
)

print('Schema validation:', 'FAIL' if schema_failed else 'PASS')
if schema_errors:
    display(schema_errors)


In [ ]:
total_rows = int(inventory['rows'].sum())
total_duplicate_rows = int(inventory['duplicate_rows'].sum())
schema_result_text = 'ผ่าน' if not schema_errors else 'ไม่ผ่าน'
overview_interpretation = (
    'ข้อมูลพร้อมเข้าสู่การตรวจคุณภาพขั้นถัดไป'
    if not schema_errors and total_duplicate_rows == 0
    else 'ควรแก้ schema หรือตรวจสอบแถวซ้ำก่อนวิเคราะห์ต่อ'
)

display(Markdown(f'''
> **สรุปข้อ 1 — ภาพรวมและ schema**
>
> - โหลดข้อมูลได้ครบ **{len(tables)} ตาราง** รวม **{total_rows:,} แถว**
> - พบแถวซ้ำทั้งแถวรวม **{total_duplicate_rows:,} แถว**
> - ผลตรวจ schema: **{schema_result_text}**
>
> {overview_interpretation}
'''))


## 2. Data dictionary

Dictionary ด้านล่างเป็นข้อมูลอ้างอิงของคอลัมน์ จึงแยกออกจากโค้ดคำนวณหลัก แต่ละตารางจะแสดงชนิดข้อมูล จำนวนค่าที่ไม่ว่าง จำนวนค่าว่าง และจำนวนค่าที่ไม่ซ้ำ


In [ ]:
COLUMN_DESCRIPTIONS = {
    # projects
    'project_id': 'รหัสโครงการ ใช้เชื่อมตาราง projects',
    'project_name': 'ชื่อโครงการ',
    'university': 'สถานศึกษาหรือทำเลหลักของโครงการ',
    'distance_to_campus_m': 'ระยะทางถึงสถานศึกษา (เมตร)',
    'total_units': 'จำนวนห้องทั้งหมดของโครงการ',
    'year_completed': 'ปีที่โครงการสร้างเสร็จ',
    'facility_count': 'จำนวนกิจกรรมหรือสิ่งอำนวยความสะดวกส่วนกลาง',
    # units
    'unit_id': 'รหัสห้อง ใช้เชื่อมตาราง units',
    'floor': 'ชั้นของห้อง',
    'size_sqm': 'ขนาดห้อง (ตารางเมตร)',
    'room_type': 'ประเภทห้อง: studio, 1BR หรือ 1BR Plus',
    'view': 'ประเภทวิว: standard, city หรือ pool',
    'furnished': 'สถานะเฟอร์นิเจอร์ครบ',
    'agent_id': 'รหัสนายหน้าที่ดูแลห้องหรือประกาศ',
    # listings
    'listing_id': 'รหัสประกาศเช่า',
    'week_listed': 'ลำดับสัปดาห์ที่ลงประกาศ เริ่มจาก 0',
    'date_listed': 'วันที่ลงประกาศ รูปแบบ YYYY-MM-DD',
    'season_listed': 'ฤดูกาล ณ วันที่ลงประกาศ',
    'asking_rent': 'ค่าเช่าล่าสุดที่ประกาศ (บาท/เดือน)',
    'first_asking_rent': 'ค่าเช่าที่ตั้งครั้งแรก (บาท/เดือน)',
    'weeks_on_market': 'จำนวนสัปดาห์ที่ประกาศอยู่ในตลาด',
    'n_viewings': 'จำนวนครั้งที่มีผู้เข้าชมห้อง',
    'leased': 'ระบุว่าประกาศปล่อยเช่าได้แล้ว',
    'week_leased': 'สัปดาห์ที่ปล่อยเช่าได้; ว่างเมื่อยังไม่ถูกเช่า',
    'tenant_segment': 'กลุ่มผู้เช่า; ว่างเมื่อยังไม่ถูกเช่า',
    'lease_months': 'ระยะสัญญาเช่า (เดือน)',
    # leases
    'lease_id': 'รหัสสัญญาเช่า',
    'week_start': 'ลำดับสัปดาห์ที่เริ่มสัญญา',
    'date_start': 'วันที่เริ่มสัญญา รูปแบบ YYYY-MM-DD',
    'week_end': 'ลำดับสัปดาห์ที่สิ้นสุดสัญญา',
    'rent': 'ค่าเช่าตามสัญญา (บาท/เดือน)',
    'is_renewal': 'ระบุว่าสัญญานี้เป็นการต่อสัญญา',
    # weekly_market
    'week': 'ลำดับสัปดาห์ เริ่มจาก 0',
    'date': 'วันที่เริ่มสัปดาห์ รูปแบบ YYYY-MM-DD',
    'season': 'ฤดูกาลของสัปดาห์',
    'n_searchers': 'จำนวนผู้ค้นหาห้องในสัปดาห์นั้น',
    'n_listings_open': 'จำนวนประกาศที่ยังเปิดอยู่',
    'n_units_leased': 'จำนวนห้องที่มีผู้เช่าอยู่',
    'occupancy': 'สัดส่วนห้องที่มีผู้เช่า ช่วง 0 ถึง 1',
    'median_asking_rent': 'มัธยฐานค่าเช่าที่ประกาศ; ว่างได้เมื่อไม่มีประกาศเปิด',
}


In [ ]:
dictionary_rows = []

for table_name, frame in tables.items():
    for column in frame.columns:
        dictionary_rows.append({
            'table': table_name,
            'column': column,
            'dtype': str(frame[column].dtype),
            'non_null': int(frame[column].notna().sum()),
            'null': int(frame[column].isna().sum()),
            'unique': int(frame[column].nunique(dropna=True)),
            'description': COLUMN_DESCRIPTIONS.get(column, ''),
        })

data_dictionary = pd.DataFrame(dictionary_rows)

# แสดงแยกทีละตาราง เพื่อลดความยาวของผลลัพธ์ในแต่ละครั้ง
for table_name in TABLES:
    table_dictionary = data_dictionary.query('table == @table_name').drop(columns='table')
    display(Markdown(f'### `{table_name}`'))
    display(table_dictionary.reset_index(drop=True))


In [ ]:
total_dictionary_columns = len(data_dictionary)
described_columns = int(data_dictionary['description'].ne('').sum())
undescribed_columns = total_dictionary_columns - described_columns

display(Markdown(f'''
> **สรุปข้อ 2 — Data dictionary**
>
> - บันทึกข้อมูล **{total_dictionary_columns:,} คอลัมน์** จาก **{data_dictionary['table'].nunique()} ตาราง**
> - มีคำอธิบายแล้ว **{described_columns:,} คอลัมน์**
> - ยังไม่มีคำอธิบาย **{undescribed_columns:,} คอลัมน์**
>
> Dictionary นี้ช่วยตรวจชนิดข้อมูล ค่าว่าง และความหมายของตัวแปรก่อนเริ่ม EDA หรือสร้าง feature
'''))


## 3. ค่าว่าง

ค่าว่างไม่ใช่ข้อผิดพลาดเสมอไป ขั้นแรกเราสรุปค่าว่างทั้งหมด แล้วจึงตรวจว่าค่าว่างเหล่านั้นสอดคล้องกับกฎทางธุรกิจหรือไม่


In [ ]:
missing_rows = []

for table_name, frame in tables.items():
    missing_counts = frame.isna().sum()

    for column, missing_count in missing_counts.items():
        if missing_count > 0:
            missing_rows.append({
                'table': table_name,
                'column': column,
                'missing': int(missing_count),
                'missing_pct': missing_count / len(frame) * 100,
            })

missing_summary = pd.DataFrame(
    missing_rows,
    columns=['table', 'column', 'missing', 'missing_pct'],
)
display(missing_summary.style.format({'missing': '{:,}', 'missing_pct': '{:.2f}%'}))


In [ ]:
outcome_columns = ['week_leased', 'tenant_segment', 'lease_months']

leased_without_outcome = (
    listings['leased']
    & listings[outcome_columns].isna().any(axis=1)
)
unleased_with_outcome = (
    ~listings['leased']
    & listings[outcome_columns].notna().any(axis=1)
)
missing_median_with_open_listings = (
    weekly_market['median_asking_rent'].isna()
    & weekly_market['n_listings_open'].gt(0)
)

record_check('Missing', 'ประกาศที่เช่าแล้วมีข้อมูลผลลัพธ์ครบ', leased_without_outcome)
record_check('Missing', 'ประกาศที่ยังไม่เช่าไม่มีข้อมูลผลลัพธ์', unleased_with_outcome)
record_check(
    'Missing',
    'median_asking_rent ว่างเฉพาะเมื่อไม่มีประกาศเปิด',
    missing_median_with_open_listings,
)

display(pd.DataFrame(quality_checks).query("area == 'Missing'"))


In [ ]:
missing_check_results = pd.DataFrame(quality_checks).query("area == 'Missing'")
missing_rule_failures = int(missing_check_results['failures'].sum())
total_missing_cells = int(missing_summary['missing'].sum())
columns_with_missing = len(missing_summary)
missing_result_text = 'สอดคล้องกับกฎทางธุรกิจ' if missing_rule_failures == 0 else 'มีค่าว่างที่ผิดกฎทางธุรกิจ'
missing_interpretation = (
    'ค่าว่างของประกาศที่ยังไม่ถูกเช่าเป็นข้อมูลที่มีความหมาย จึงไม่ควรลบหรือเติมค่าโดยอัตโนมัติ'
    if missing_rule_failures == 0
    else 'ควรตรวจแถวที่ไม่ผ่านกฎก่อนตัดสินใจลบหรือเติมค่า'
)

display(Markdown(f'''
> **สรุปข้อ 3 — ค่าว่าง**
>
> - พบค่าว่างรวม **{total_missing_cells:,} cells** ใน **{columns_with_missing} คอลัมน์**
> - จำนวนแถวที่ไม่ผ่านกฎค่าว่าง: **{missing_rule_failures:,} แถว**
> - ข้อสรุป: **{missing_result_text}**
>
> {missing_interpretation}
'''))


## 4. Primary key และ foreign key

- **Primary key** ต้องระบุแต่ละแถวได้โดยไม่ซ้ำ
- **Foreign key** ต้องอ้างถึงรหัสที่มีอยู่จริงในตารางหลัก


In [ ]:
primary_keys = {
    'projects': ['project_id'],
    'units': ['unit_id'],
    'listings': ['listing_id'],
    'leases': ['lease_id'],
    'weekly_market': ['project_id', 'week'],
}

for table_name, key_columns in primary_keys.items():
    duplicate_key = tables[table_name].duplicated(key_columns)
    key_label = ', '.join(key_columns)

    record_check(
        area='Primary key',
        check=f'{table_name}: {key_label} ไม่ซ้ำ',
        failure_mask=duplicate_key,
    )


In [ ]:
valid_project_ids = set(projects['project_id'])
valid_unit_ids = set(units['unit_id'])

foreign_key_rules = [
    ('units.project_id → projects', ~units['project_id'].isin(valid_project_ids)),
    ('listings.project_id → projects', ~listings['project_id'].isin(valid_project_ids)),
    ('listings.unit_id → units', ~listings['unit_id'].isin(valid_unit_ids)),
    ('leases.project_id → projects', ~leases['project_id'].isin(valid_project_ids)),
    ('leases.unit_id → units', ~leases['unit_id'].isin(valid_unit_ids)),
    ('weekly_market.project_id → projects', ~weekly_market['project_id'].isin(valid_project_ids)),
]

for rule_name, invalid_foreign_key in foreign_key_rules:
    record_check('Foreign key', rule_name, invalid_foreign_key)

display(pd.DataFrame(quality_checks).query("area in ['Primary key', 'Foreign key']"))


In [ ]:
key_check_results = pd.DataFrame(quality_checks).query(
    "area in ['Primary key', 'Foreign key']"
)
failed_key_checks = int(key_check_results['status'].eq('FAIL').sum())
invalid_key_rows = int(key_check_results['failures'].sum())
key_result_text = 'ผ่านทุกกฎ' if failed_key_checks == 0 else 'พบปัญหาที่ต้องแก้ไข'

display(Markdown(f'''
> **สรุปข้อ 4 — Primary key และ foreign key**
>
> - ตรวจทั้งหมด **{len(key_check_results)} กฎ**
> - กฎที่ไม่ผ่าน: **{failed_key_checks} กฎ**
> - แถวที่มีคีย์ผิดเงื่อนไขรวม: **{invalid_key_rows:,} แถว**
> - ข้อสรุป: **{key_result_text}**
>
> เมื่อทุกกฎผ่าน แต่ละแถวจะมีรหัสไม่ซ้ำและสามารถเชื่อมไปยังตารางหลักได้อย่างถูกต้อง
'''))


## 5. ความสอดคล้องข้ามตาราง

แม้ foreign key จะมีอยู่จริง แต่ข้อมูลประกอบ เช่น ชื่อโครงการ ชั้น หรือขนาดห้อง อาจไม่ตรงกับตารางหลัก จึงต้องตรวจค่าเหล่านี้เพิ่มเติม


In [ ]:
unit_project_lookup = units.set_index('unit_id')['project_id']

listing_project_mismatch = listings['project_id'].ne(
    listings['unit_id'].map(unit_project_lookup)
)
lease_project_mismatch = leases['project_id'].ne(
    leases['unit_id'].map(unit_project_lookup)
)

record_check(
    'Cross-table',
    'listings.project_id ตรงกับ units.project_id',
    listing_project_mismatch,
)
record_check(
    'Cross-table',
    'leases.project_id ตรงกับ units.project_id',
    lease_project_mismatch,
)


In [ ]:
project_lookup = projects.set_index('project_id')
project_columns = ['project_name', 'university', 'distance_to_campus_m', 'facility_count']

for column in project_columns:
    expected_value = listings['project_id'].map(project_lookup[column])
    value_mismatch = listings[column].ne(expected_value)
    record_check(
        'Cross-table',
        f'listings.{column} ตรงกับ projects.{column}',
        value_mismatch,
    )

unit_lookup = units.set_index('unit_id')
unit_columns = ['floor', 'size_sqm', 'room_type', 'view', 'furnished']

for column in unit_columns:
    expected_value = listings['unit_id'].map(unit_lookup[column])
    value_mismatch = listings[column].ne(expected_value)
    record_check(
        'Cross-table',
        f'listings.{column} ตรงกับ units.{column}',
        value_mismatch,
    )

actual_unit_count = units.groupby('project_id').size().reindex(
    project_lookup.index,
    fill_value=0,
)
unit_count_mismatch = actual_unit_count.ne(project_lookup['total_units'])
record_check(
    'Cross-table',
    'จำนวน units ต่อโครงการตรงกับ projects.total_units',
    unit_count_mismatch,
)

display(pd.DataFrame(quality_checks).query("area == 'Cross-table'"))


In [ ]:
cross_table_results = pd.DataFrame(quality_checks).query("area == 'Cross-table'")
failed_cross_table_checks = int(cross_table_results['status'].eq('FAIL').sum())
cross_table_mismatches = int(cross_table_results['failures'].sum())
cross_table_result_text = 'ข้อมูลตรงกันทุกตาราง' if failed_cross_table_checks == 0 else 'พบข้อมูลที่ไม่ตรงกัน'
cross_table_interpretation = (
    'รหัสโครงการ รายละเอียดห้อง และจำนวนห้องต่อโครงการสามารถใช้อ้างอิงร่วมกันได้'
    if failed_cross_table_checks == 0
    else 'ควรแก้ค่าที่ไม่ตรงกันก่อน merge ตารางหรือสร้าง feature ข้ามตาราง'
)

display(Markdown(f'''
> **สรุปข้อ 5 — ความสอดคล้องข้ามตาราง**
>
> - ตรวจทั้งหมด **{len(cross_table_results)} กฎ**
> - กฎที่ไม่ผ่าน: **{failed_cross_table_checks} กฎ**
> - ค่าที่ไม่ตรงกันรวม: **{cross_table_mismatches:,} รายการ**
> - ข้อสรุป: **{cross_table_result_text}**
>
> {cross_table_interpretation}
'''))


## 6. ช่วงค่า หมวดหมู่ และตรรกะภายในตาราง

กฎส่วนนี้มาจากความหมายของข้อมูล ไม่ใช้ IQR ตัดค่าที่ดูสุดโต่งโดยอัตโนมัติ เพราะค่าจริงบางค่าอาจอยู่ห่างจากข้อมูลส่วนใหญ่แต่ยังถูกต้อง


### 6.1 ตาราง `units`

ตรวจชั้น ขนาดห้อง ประเภทห้อง และประเภทวิวตามขอบเขตที่กำหนดไว้ในค่าคงที่


In [ ]:
invalid_floor = ~units['floor'].between(*VALID_FLOOR_RANGE)
invalid_size = ~units['size_sqm'].between(*VALID_SIZE_RANGE_SQM)
invalid_room_type = ~units['room_type'].isin(VALID_ROOM_TYPES)
invalid_view = ~units['view'].isin(VALID_VIEWS)

record_check('Domain', 'units.floor อยู่ในช่วง 1–8', invalid_floor)
record_check('Domain', 'units.size_sqm อยู่ในช่วง 20–50', invalid_size)
record_check('Domain', 'units.room_type อยู่ในหมวดที่กำหนด', invalid_room_type)
record_check('Domain', 'units.view อยู่ในหมวดที่กำหนด', invalid_view)


### 6.2 ตาราง `listings`

ตรวจค่าเช่า จำนวนสัปดาห์ จำนวนผู้เข้าชม ฤดูกาล และความสัมพันธ์ระหว่างวันลงประกาศกับวันปล่อยเช่า


In [ ]:
non_positive_rent = (
    listings[['asking_rent', 'first_asking_rent']] <= 0
).any(axis=1)
negative_activity = (
    listings[['weeks_on_market', 'n_viewings']] < 0
).any(axis=1)
invalid_season = ~listings['season_listed'].isin(VALID_SEASONS)

leased_listings = listings['leased']
expected_week_leased = listings['week_listed'] + listings['weeks_on_market']
invalid_week_leased = (
    leased_listings
    & listings['week_leased'].ne(expected_week_leased)
)

record_check('Domain', 'ค่าเช่าใน listings เป็นบวก', non_positive_rent)
record_check('Domain', 'weeks_on_market และ n_viewings ไม่ติดลบ', negative_activity)
record_check('Domain', 'season_listed อยู่ในหมวดที่กำหนด', invalid_season)
record_check(
    'Logic',
    'week_leased = week_listed + weeks_on_market เมื่อเช่าแล้ว',
    invalid_week_leased,
)


In [ ]:
alternative_listings = listings.loc[
    listings['listing_id'].str.endswith('-B')
].copy()
alternative_listings['base_listing_id'] = (
    alternative_listings['listing_id'].str[:-2]
)
shared_event_columns = [
    'unit_id', 'project_id', 'week_listed', 'date_listed',
    'first_asking_rent', 'weeks_on_market', 'leased', 'week_leased',
]
alternative_pairs = alternative_listings.merge(
    listings.set_index('listing_id')[shared_event_columns],
    left_on='base_listing_id',
    right_index=True,
    how='left',
    validate='one_to_one',
    suffixes=('', '_base'),
)

left_values = alternative_pairs[shared_event_columns].reset_index(drop=True)
right_values = alternative_pairs[
    [f'{column}_base' for column in shared_event_columns]
].set_axis(shared_event_columns, axis=1).reset_index(drop=True)
same_event = (
    left_values.eq(right_values)
    | (left_values.isna() & right_values.isna())
).all(axis=1)
invalid_alternative_pair = ~same_event

changed_price_count = int((
    alternative_pairs['asking_rent']
    != alternative_pairs['base_listing_id'].map(
        listings.set_index('listing_id')['asking_rent']
    )
).sum())
changed_agent_count = int((
    alternative_pairs['agent_id']
    != alternative_pairs['base_listing_id'].map(
        listings.set_index('listing_id')['agent_id']
    )
).sum())
alternative_note = (
    f'พบ {len(alternative_pairs):,} แถว; เปลี่ยน asking_rent '
    f'{changed_price_count:,} แถว และ agent_id {changed_agent_count:,} แถว '
    'จึงเก็บใน source เพื่อ audit แต่ตัดออกจาก Modeling Dataset'
)

record_check(
    area='Logic',
    check='รายการ -B จับคู่กับประกาศหลักและใช้เหตุการณ์เดียวกัน',
    failure_mask=invalid_alternative_pair,
    note=alternative_note,
)


### 6.3 ตาราง `leases`

ตรวจระยะสัญญา การต่อสัญญา และค่าเช่าตามสัญญา


In [ ]:
expected_lease_weeks = leases['lease_months'].map(LEASE_WEEKS_BY_MONTHS)
actual_lease_weeks = leases['week_end'] - leases['week_start']

invalid_lease_duration = actual_lease_weeks.ne(expected_lease_weeks)
invalid_renewal_flag = leases['is_renewal'].ne(
    leases['tenant_segment'].eq('renewal')
)
non_positive_lease_rent = leases['rent'].le(0)

record_check('Logic', 'ระยะสัญญาตรงกับ lease_months', invalid_lease_duration)
record_check('Logic', 'is_renewal ตรงกับ tenant_segment=renewal', invalid_renewal_flag)
record_check('Domain', 'leases.rent เป็นบวก', non_positive_lease_rent)


### 6.4 ตาราง `weekly_market`

ตรวจช่วง occupancy จำนวนที่ไม่ติดลบ และคำนวณ occupancy ซ้ำจากตารางหลักเพื่อยืนยันผล


In [ ]:
invalid_occupancy_range = ~weekly_market['occupancy'].between(0, 1)
negative_weekly_counts = (
    weekly_market[['n_searchers', 'n_listings_open', 'n_units_leased']] < 0
).any(axis=1)

project_total_units = weekly_market['project_id'].map(project_lookup['total_units'])
expected_occupancy = weekly_market['n_units_leased'] / project_total_units
occupancy_mismatch = ~np.isclose(
    weekly_market['occupancy'],
    expected_occupancy,
)

record_check('Domain', 'occupancy อยู่ในช่วง 0–1', invalid_occupancy_range)
record_check('Domain', 'จำนวนใน weekly_market ไม่ติดลบ', negative_weekly_counts)
record_check(
    'Logic',
    'occupancy = n_units_leased / total_units',
    occupancy_mismatch,
)
display(pd.DataFrame(quality_checks).query("area in ['Domain', 'Logic']"))


In [ ]:
domain_logic_results = pd.DataFrame(quality_checks).query(
    "area in ['Domain', 'Logic']"
)
failed_domain_checks = int(domain_logic_results['status'].eq('FAIL').sum())
warning_domain_checks = int(domain_logic_results['status'].eq('WARN').sum())

unit_issue_count = int(
    invalid_floor.sum()
    + invalid_size.sum()
    + invalid_room_type.sum()
    + invalid_view.sum()
)
listing_issue_count = int(
    non_positive_rent.sum()
    + negative_activity.sum()
    + invalid_season.sum()
    + invalid_week_leased.sum()
)
lease_issue_count = int(
    invalid_lease_duration.sum()
    + invalid_renewal_flag.sum()
    + non_positive_lease_rent.sum()
)
weekly_issue_count = int(
    invalid_occupancy_range.sum()
    + negative_weekly_counts.sum()
    + occupancy_mismatch.sum()
)
domain_interpretation = (
    'กฎทั้งหมดผ่าน และรายการ -B ถูกระบุเป็นข้อมูลทางเลือกของเหตุการณ์เดียวกันอย่างชัดเจน'
    if failed_domain_checks == 0
    else 'ควรแก้ค่าที่ผิดช่วงหรือผิดตรรกะก่อนทำ EDA และสร้าง feature'
)

display(Markdown(f'''
> **สรุปข้อ 6 — ช่วงค่า หมวดหมู่ และตรรกะ**
>
> - `units`: พบค่าผิดเงื่อนไข **{unit_issue_count:,} รายการ**
> - `listings`: พบค่าผิดเงื่อนไข **{listing_issue_count:,} รายการ**; รายการทางเลือก `-B` **{len(alternative_pairs):,} แถว**
> - `leases`: พบค่าผิดเงื่อนไข **{lease_issue_count:,} รายการ**
> - `weekly_market`: พบค่าผิดเงื่อนไข **{weekly_issue_count:,} รายการ**
> - รวมกฎที่ `FAIL`: **{failed_domain_checks} กฎ**, `WARN`: **{warning_domain_checks} กฎ**
>
> {domain_interpretation}
'''))


## 7. วันที่และลำดับสัปดาห์

สัปดาห์ที่ 0 เริ่มวันที่ 3 มกราคม 2023 ดังนั้นวันที่ที่คาดหวังคำนวณจาก

`วันที่เริ่มต้น + (เลขสัปดาห์ × 7 วัน)`


In [ ]:
date_rules = [
    ('listings', listings, 'date_listed', 'week_listed'),
    ('leases', leases, 'date_start', 'week_start'),
    ('weekly_market', weekly_market, 'date', 'week'),
]
date_results = []

for table_name, frame, date_column, week_column in date_rules:
    parsed_date = pd.to_datetime(frame[date_column], errors='coerce')
    expected_date = DATA_START_DATE + pd.to_timedelta(
        frame[week_column] * DAYS_PER_WEEK,
        unit='D',
    )

    invalid_date = parsed_date.isna()
    date_mismatch = parsed_date.notna() & parsed_date.ne(expected_date)

    record_check('Date', f'{table_name}.{date_column} อ่านเป็นวันที่ได้', invalid_date)
    record_check('Date', f'{table_name}: วันที่ตรงกับเลขสัปดาห์', date_mismatch)

    date_results.append({
        'table': table_name,
        'invalid_dates': int(invalid_date.sum()),
        'week_date_mismatches': int(date_mismatch.sum()),
        'min_date': parsed_date.min().date(),
        'max_date': parsed_date.max().date(),
    })

date_summary_table = pd.DataFrame(date_results)
display(date_summary_table)


In [ ]:
total_invalid_dates = int(date_summary_table['invalid_dates'].sum())
total_date_mismatches = int(date_summary_table['week_date_mismatches'].sum())
first_date = date_summary_table['min_date'].min()
last_date = date_summary_table['max_date'].max()
date_result_text = 'วันที่ถูกต้องและตรงกับเลขสัปดาห์' if total_invalid_dates + total_date_mismatches == 0 else 'พบวันที่ที่ต้องตรวจสอบ'

display(Markdown(f'''
> **สรุปข้อ 7 — วันที่และลำดับสัปดาห์**
>
> - วันที่ที่แปลงค่าไม่ได้: **{total_invalid_dates:,} รายการ**
> - วันที่ที่ไม่ตรงกับเลขสัปดาห์: **{total_date_mismatches:,} รายการ**
> - ช่วงวันที่รวม: **{first_date} ถึง {last_date}**
> - ข้อสรุป: **{date_result_text}**
'''))


## 8. เปรียบเทียบ raw กับ processed

ตรวจว่าการทำความสะอาดรักษาจำนวนแถวไว้ และแก้ปัญหาหลักใน `listings` โดยไม่แก้ไฟล์ raw


In [ ]:
comparison_rows = []

for table_name in TABLES:
    raw_frame = raw_tables[table_name]
    processed_frame = tables[table_name]
    row_count_changed = len(raw_frame) != len(processed_frame)

    comparison_rows.append({
        'table': table_name,
        'raw_rows': len(raw_frame),
        'processed_rows': len(processed_frame),
        'row_count_preserved': not row_count_changed,
        'raw_missing_cells': int(raw_frame.isna().sum().sum()),
        'processed_missing_cells': int(processed_frame.isna().sum().sum()),
    })

    record_check(
        'Raw→processed',
        f'{table_name}: จำนวนแถวไม่เปลี่ยน',
        row_count_changed,
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison.style.format({
    'raw_rows': '{:,}',
    'processed_rows': '{:,}',
    'raw_missing_cells': '{:,}',
    'processed_missing_cells': '{:,}',
}))


In [ ]:
raw_listings = raw_tables['listings']
processed_listings = tables['listings']

cleanup_rules = [
    (
        'size_sqm นอกช่วง 20–50',
        ~raw_listings['size_sqm'].between(*VALID_SIZE_RANGE_SQM),
        ~processed_listings['size_sqm'].between(*VALID_SIZE_RANGE_SQM),
    ),
    (
        'asking_rent นอกช่วงตรวจสอบ 5,000–30,000',
        ~raw_listings['asking_rent'].between(*RENT_REVIEW_RANGE),
        ~processed_listings['asking_rent'].between(*RENT_REVIEW_RANGE),
    ),
    (
        'room_type มีช่องว่างหัวหรือท้าย',
        raw_listings['room_type'].ne(raw_listings['room_type'].str.strip()),
        processed_listings['room_type'].ne(processed_listings['room_type'].str.strip()),
    ),
    (
        'view ไม่ใช่ตัวพิมพ์เล็ก',
        raw_listings['view'].ne(raw_listings['view'].str.lower()),
        processed_listings['view'].ne(processed_listings['view'].str.lower()),
    ),
    (
        'furnished หาย',
        raw_listings['furnished'].isna(),
        processed_listings['furnished'].isna(),
    ),
    (
        'date_listed ไม่ใช่รูปแบบ YYYY-MM-DD',
        ~raw_listings['date_listed'].astype('string').str.fullmatch(
            r'[0-9]{4}-[0-9]{2}-[0-9]{2}'
        ),
        ~processed_listings['date_listed'].astype('string').str.fullmatch(
            r'[0-9]{4}-[0-9]{2}-[0-9]{2}'
        ),
    ),
]

listing_cleanup = pd.DataFrame([
    {
        'issue': issue,
        'raw': int(raw_issue.sum()),
        'processed': int(processed_issue.sum()),
    }
    for issue, raw_issue, processed_issue in cleanup_rules
])

display(listing_cleanup.style.format({'raw': '{:,}', 'processed': '{:,}'}))


In [ ]:
tables_with_changed_rows = int((~comparison['row_count_preserved']).sum())
raw_issue_count = int(listing_cleanup['raw'].sum())
processed_issue_count = int(listing_cleanup['processed'].sum())
fixed_issue_count = raw_issue_count - processed_issue_count
comparison_interpretation = (
    'กระบวนการทำความสะอาดแก้ปัญหาที่ตรวจพบและยังรักษาจำนวนแถวของทุกตารางไว้'
    if tables_with_changed_rows == 0 and processed_issue_count == 0
    else 'ควรตรวจขั้นตอนทำความสะอาดเพิ่มเติม เพราะยังมีปัญหาหรือจำนวนแถวเปลี่ยนไป'
)

display(Markdown(f'''
> **สรุปข้อ 8 — เปรียบเทียบ raw กับ processed**
>
> - ตารางที่จำนวนแถวเปลี่ยน: **{tables_with_changed_rows} ตาราง**
> - ปัญหาใน `listings` ก่อนทำความสะอาด: **{raw_issue_count:,} รายการ**
> - ปัญหาหลังทำความสะอาด: **{processed_issue_count:,} รายการ**
> - แก้ปัญหาได้รวม: **{fixed_issue_count:,} รายการ**
>
> {comparison_interpretation}
'''))


## 9. สรุปผลตรวจทั้งหมด

ตารางแรกเรียง `FAIL` และ `WARN` ไว้ด้านบน เพื่อให้เห็นปัญหาที่ต้องจัดการก่อน


In [ ]:
check_results = pd.DataFrame(quality_checks)
status_order = pd.CategoricalDtype(['FAIL', 'WARN', 'PASS'], ordered=True)
check_results['status'] = check_results['status'].astype(status_order)
check_results = check_results.sort_values(
    ['status', 'area', 'check']
).reset_index(drop=True)

display(check_results)

n_fail = int(check_results['status'].eq('FAIL').sum())
n_warn = int(check_results['status'].eq('WARN').sum())
unleased_count = int((~listings['leased']).sum())
missing_median_count = int(weekly_market['median_asking_rent'].isna().sum())
projects_without_listings = sorted(
    valid_project_ids - set(listings['project_id'])
)

overall_status = 'PASS' if n_fail == 0 else 'FAIL'

display(Markdown(f'''
### ผลรวม: **{overall_status}**

- ข้อผิดพลาดร้ายแรง: **{n_fail}**
- ข้อควรติดตาม: **{n_warn}**
- Primary key และ foreign key ของทั้ง 5 ตารางผ่านการตรวจสอบ
- ค่าว่างในผลลัพธ์ของ `listings` จำนวน **{unleased_count:,} แถว** เกิดจากประกาศที่ยังไม่ถูกเช่า จึงต้องเก็บไว้
- `median_asking_rent` ว่าง **{missing_median_count:,} แถว** และเกิดเมื่อไม่มีประกาศเปิด
- โครงการที่ไม่มี listings ในช่วงข้อมูลคือ project_id **{projects_without_listings}**
- รายการ `-B` จำนวน **{len(alternative_pairs):,} แถว** จับคู่กับประกาศหลักได้ครบ และจะไม่นับซ้ำใน Modeling Dataset

**ข้อสรุป:** ข้อมูล processed พร้อมใช้สำหรับ EDA ขั้นถัดไป โดยต้องเก็บประกาศที่ยังไม่ถูกเช่าเพื่อหลีกเลี่ยง selection bias ตัดรายการทางเลือก `-B` ออกจากชุดฝึกเพื่อไม่ให้นับเหตุการณ์ซ้ำ และใช้เฉพาะข้อมูลที่ทราบ ณ วันลงประกาศเพื่อป้องกัน data leakage
'''))
